In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here: Read the dataset Q1_data.csv using read_csv()
import os
import pandas as pd
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)




In [ ]:
# Task 2: Write your code here:  Inspect the first few rows using head()

df.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()
df.info()


In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time)
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here: (Drop the 'Order_ID' column from the data)
df = df.drop(columns="Order_ID", axis=1)
df

In [ ]:
# Task 2: Write your code here: (Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values))
def check_missing_values(df):
  df.isnull()
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")

  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
print("\n\n\n")

def check_missing_percentage(df_func):

  missing_percentage = (df_func.isnull().sum() / len(df_func)) * 100
  missing_data = pd.DataFrame({
      'Column': missing_percentage.index,
      'Missing_Percentage': missing_percentage.values
  })
  missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

  print("Missing Data Analysis:",missing_data)


df_clean=df.copy() #new name

#categorical cleaning
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in [categorical_cols]:
    df_clean[col] = df_clean[col].fillna('unknown')
check_missing_percentage(df_clean)
#numrical cleaning
numrical_cols = df.select_dtypes(exclude=["object"]).columns
for col in [numrical_cols]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Missing percentage columns:\n---------------------------")
check_missing_percentage(df_clean)


In [ ]:
# Task 3: Write your code here: (Check and remove duplicates if any exist)
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(duplicates)
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True) #inplace is used to change the data frame if false then we need to save it in new data frame or in the same to change the data
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean

In [ ]:
# Task 5: Write your code here: (Apply feature scaling for all features (Use StandardScaler))
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(exclude=["object"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here: (Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed))
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

print("################### THE DATA IS IMBALANCED AS THE FIGURE SHOWS ######################")

In [ ]:
# Task 1: Write your code here: (Split the dataset into features (X) and target (y))
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here: (Use the correct split: KFold OR StratifiedKFold + Train a RandomForest model + Evaluate using MAE (Mean Absolute Error) ONLY+Print the averaged score across all folds)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error as sklearn_mae, mean_absolute_error, r2_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(5, shuffle=True, random_state=42)



lr_mae = []
lr_rmse = []
lr_r2 = []
sum_score=0.0
models = {"Random Forest Regressor": RandomForestRegressor(n_estimators=200)}

for name in models:

  for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    for model_name, model in models.items():
      print(f"Training {model_name}...")

      # Train
      model.fit(X_train, y_train)

      # Predict
      y_pred = model.predict(X_test)

      # Calculate metrics
      mae = mean_absolute_error(y_test, y_pred)
      rmae = np.sqrt(mae)
      r2 = r2_score(y_test, y_pred)

      # Store results
      lr_mae.append(mae)
      lr_r2.append(r2)

print("MEAN_ABSLUTE ERROR= ", lr_mae)
print("SCORE= ",lr_r2)
for score_per_fold in lr_r2:
  sum_score+=score_per_fold
print("AVRAGED SCORE= ",sum_score/n_splits)



In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: